# **Project Overview**

# **Anime Recommendation System**

Building an Anime Recommendation System using Cosine Similarity based on features like:

->Genre

->Rating

->Episodes

->Popularity (members)

This is a content-based filtering system (not collaborative filtering).


In [65]:
# Step 1: Import Libraries
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

In [66]:
# Step 2: Load Dataset
df = pd.read_csv("/content/anime.csv")
df.head()


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [67]:
df

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266
...,...,...,...,...,...,...,...
12289,9316,Toushindai My Lover: Minami tai Mecha-Minami,Hentai,OVA,1,4.15,211
12290,5543,Under World,Hentai,OVA,1,4.28,183
12291,5621,Violence Gekiga David no Hoshi,Hentai,OVA,4,4.88,219
12292,6133,Violence Gekiga Shin David no Hoshi: Inma Dens...,Hentai,OVA,1,4.98,175


In [68]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [69]:
df.isnull().sum()

,0
anime_id,0
name,0
genre,62
type,25
episodes,0
rating,230
members,0


In [70]:
print(df.describe())
print(df['genre'].head())

           anime_id        rating       members
count  12294.000000  12064.000000  1.229400e+04
mean   14058.221653      6.473902  1.807134e+04
std    11455.294701      1.026746  5.482068e+04
min        1.000000      1.670000  5.000000e+00
25%     3484.250000      5.880000  2.250000e+02
50%    10260.500000      6.570000  1.550000e+03
75%    24794.500000      7.180000  9.437000e+03
max    34527.000000     10.000000  1.013917e+06
0                 Drama, Romance, School, Supernatural
1    Action, Adventure, Drama, Fantasy, Magic, Mili...
2    Action, Comedy, Historical, Parody, Samurai, S...
3                                     Sci-Fi, Thriller
4    Action, Comedy, Historical, Parody, Samurai, S...
Name: genre, dtype: object


In [74]:
print(df[['rating', 'episodes', 'members']].isnull().sum())

rating      0
episodes    0
members     0
dtype: int64


In [78]:
# Step 3: Data Preprocessing
# Handle missing values
df['genre'] = df['genre'].fillna('')
df['rating'] = df['rating'].fillna(df['rating'].mean())

df['episodes'] = df['episodes'].replace('Unknown', np.nan)
# Convert types before calculating median or filling NaN
df['episodes'] = df['episodes'].astype(float)
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

df['members'] = df['members'].fillna(df['members'].median())


# Step 4: Feature Extraction
# Vectorize genre
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(tokenizer=lambda x: x.split(','), token_pattern=None)
genre_matrix = cv.fit_transform(df['genre'])

# Normalize numerical features
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
num_features = scaler.fit_transform(df[['rating', 'episodes', 'members']])

# Combine
from scipy.sparse import hstack
feature_matrix = hstack([genre_matrix, num_features])

# Step:5 Compute Cosine Similarity

from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(feature_matrix)

In [84]:
# Step 6: Recommendation Function
def recommend_anime(title, n=10):
    idx = df[df['name'] == title].index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:n+1]

    anime_indices = [i[0] for i in sim_scores]

    return df['name'].iloc[anime_indices].values

In [85]:
print(recommend_anime("Naruto"))

['Naruto: Shippuuden' 'Naruto: Shippuuden Movie 4 - The Lost Tower'
 'Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsugu Mono'
 'Boruto: Naruto the Movie' 'Naruto x UT'
 'Naruto Soyokazeden Movie: Naruto to Mashin to Mitsu no Onegai Dattebayo!!'
 'Boruto: Naruto the Movie - Naruto ga Hokage ni Natta Hi'
 'Naruto Shippuuden: Sunny Side Battle' 'Katekyo Hitman Reborn!'
 'Kyutai Panic Adventure!']


In [91]:
# Step 7: Threshold-Based Recommendations
def recommend_with_threshold(title, threshold=0.5):
    idx = df[df['name'] == title].index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))

    filtered = [i for i in sim_scores if i[1] > threshold]

    filtered = sorted(filtered, key=lambda x: x[1], reverse=True)

    anime_indices = [i[0] for i in filtered[1:]]

    return df['name'].iloc[anime_indices]

In [94]:
for anime in recommend_with_threshold("Naruto", 0.6):
    print(anime)

Naruto: Shippuuden
Naruto: Shippuuden Movie 4 - The Lost Tower
Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsugu Mono
Boruto: Naruto the Movie
Naruto x UT
Naruto Soyokazeden Movie: Naruto to Mashin to Mitsu no Onegai Dattebayo!!
Boruto: Naruto the Movie - Naruto ga Hokage ni Natta Hi
Naruto Shippuuden: Sunny Side Battle
Katekyo Hitman Reborn!
Kyutai Panic Adventure!
Battle Spirits: Ryuuko no Ken
Dragon Ball Z
Dragon Ball Kai
Bleach
Dragon Ball Super
Medaka Box
Tenjou Tenge
Medaka Box Abnormal
Dragon Ball Kai (2014)
Dragon Ball Z Movie 15: Fukkatsu no F
Dragon Ball GT: Goku Gaiden! Yuuki no Akashi wa Suushinchuu
Dragon Ball Z Movie 11: Super Senshi Gekiha!! Katsu no wa Ore da
Dragon Ball Z: Summer Vacation Special
Dragon Ball Z: Atsumare! Gokuu World
Boku no Hero Academia
Shijou Saikyou no Deshi Kenichi
Bleach Movie 3: Fade to Black - Kimi no Na wo Yobu
Bleach Movie 4: Jigoku-hen
Naruto: Shippuuden Movie 6 - Road to Ninja
The Last: Naruto the Movie
Shijou Saikyou no Deshi Kenichi OVA
Rek